In [1]:
!python -V

Python 3.12.1


In [1]:
!mlflow --version

mlflow, version 3.3.1


In [3]:
import pandas as pd

In [15]:
import pickle

In [14]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [16]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/Users/volodymyr_shvachko/Project/MLOps/mlops-zoomcamp-lab/02-Exp-Track/mlruns/2', creation_time=1756038352552, experiment_id='2', last_update_time=1756038352552, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [8]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [4]:
df_file = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')
print(len(df_file))

3403766


In [17]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')
#print(len(df_train))

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression

categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

train_dicts = df_train[categorical].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

target = 'duration'
y_train = df_train[target].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)
print(lr.intercept_)

24.776302895683436


In [26]:
y_pred = lr.predict(X_train)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [ ]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

#train_dicts = df_train[categorical + numerical].to_dict(orient='records')
train_dicts = df_train[categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [20]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [22]:
import xgboost as xgb

In [23]:
from pathlib import Path

In [24]:
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [27]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=30,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/usr/local/lib/python3.11/site-packages/xgboost/callback.py:386: UserWarning: [21:50:07] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:245: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:9.64799
[1]	validation-rmse:9.28113
[2]	validation-rmse:8.96030
[3]	validation-rmse:8.65189
[4]	validation-rmse:8.41694
[5]	validation-rmse:8.22130
[6]	validation-rmse:8.05184
[7]	validation-rmse:7.92028
[8]	validation-rmse:7.74802
[9]	validation-rmse:7.65453
[10]	validation-rmse:7.55392
[11]	validation-rmse:7.48519
[12]	validation-rmse:7.41252
[13]	validation-rmse:7.36213
[14]	validation-rmse:7.31841
[15]	validation-rmse:7.21723
[16]	validation-rmse:7.18553
[17]	validation-rmse:7.14551
[18]	validation-rmse:7.12000
[19]	validation-rmse:7.02170
[20]	validation-rmse:7.00124
[21]	validation-rmse:6.91199
[22]	validation-rmse:6.89479
[23]	validation-rmse:6.87790
[24]	validation-rmse:6.86315
[25]	validation-rmse:6.84850
[26]	validation-rmse:6.78254
[27]	validation-rmse:6.77023
[28]	validation-rmse:6.75537
[29]	validation-rmse:6.74021


2025/08/28 21:50:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/usr/local/lib/python3.11/site-packages/mlflow/xgboost/__init__.py:169: UserWarning: [21:50:22] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)
2025/08/28 21:50:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run awesome-lark-795 at: http://localhost:5000/#/experiments/2/runs/50d1eef39d7a42f0bfc12d6bff1db97b
🧪 View experiment at: http://localhost:5000/#/experiments/2
